## Cameras

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

In [2]:
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

In [3]:
Cameras = []
seen_links = set() 

In [4]:
START_URL = "https://www.amazon.eg/s?i=electronics&rh=n%3A21832873031&s=popularity-rank&fs=true&language=en&ref=lp_21832873031_sar"

driver = webdriver.Chrome(options=options)
driver.get(START_URL)

## -- Accept Cookies --
try:
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.ID, "sp-cc-accept"))
    ).click()
except TimeoutException:
    pass

## -- Click "See all results" only if it exists (category pages only) --
try:
    WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.ID, "apb-desktop-browse-search-see-all"))
    ).click()
except TimeoutException:
    pass

In [5]:
## -- Scraping --
page = 1

while True:
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
            )
        )
    except TimeoutException:
        print("No results found on this page (maybe a CAPTCHA?). Stopping.")
        break

    cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
    page_products = 0

    for card in cards:
        try:
            Name = card.find_element(By.CSS_SELECTOR, 'h2').get_attribute('aria-label')
            Link = card.find_element(By.CSS_SELECTOR, 'a.a-link-normal[href*="/dp/"]').get_attribute('href')
            Image = card.find_element(By.CSS_SELECTOR, 'img.s-image').get_attribute('src')
        except NoSuchElementException:
            continue

        clean_link = Link.split("?")[0]
        if clean_link in seen_links:
            continue
        seen_links.add(clean_link)

        try:
            Rating = card.find_element(By.CSS_SELECTOR, '[data-cy="reviews-block"] span.a-size-small.a-color-base').text
        except NoSuchElementException:
            Rating = "Not Available"

        try:
            Reviews = card.find_element(By.CSS_SELECTOR, '[data-cy="reviews-block"] a[aria-label$="ratings"]').get_attribute('aria-label')
        except NoSuchElementException:
            Reviews = "Not Available"

        try:
            Price = card.find_element(By.CLASS_NAME, 'a-price-whole').text
        except NoSuchElementException:
            Price = "Product Not Available"

        Cameras.append({
            'Name': Name,
            'Link': clean_link,
            'Image': Image,
            'Rating': Rating,
            'Reviews': Reviews,
            'Price': Price
        })
        page_products += 1

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Cameras)}"
    )

    try:
        next_page = driver.find_element(By.CSS_SELECTOR, 'a.s-pagination-next')

        if 's-pagination-disabled' in next_page.get_attribute('class'):
            print("Scraping finished. No more pages available.")
            break

        next_page.click()
        page += 1
        time.sleep(5)

    except NoSuchElementException:
        print("Next page button not found. Check the pagination selector.")
        break

driver.quit()

Page 1 finished successfully. Products collected on this page: 24. Total products collected: 24
Page 2 finished successfully. Products collected on this page: 24. Total products collected: 48
Page 3 finished successfully. Products collected on this page: 24. Total products collected: 72
Page 4 finished successfully. Products collected on this page: 24. Total products collected: 96
Page 5 finished successfully. Products collected on this page: 24. Total products collected: 120
Page 6 finished successfully. Products collected on this page: 24. Total products collected: 144
Page 7 finished successfully. Products collected on this page: 24. Total products collected: 168
Page 8 finished successfully. Products collected on this page: 24. Total products collected: 192
Page 9 finished successfully. Products collected on this page: 24. Total products collected: 216
Page 10 finished successfully. Products collected on this page: 24. Total products collected: 240
Page 11 finished successfully. Pr

In [6]:
df_Cameras = pd.DataFrame(Cameras)
df_Cameras.head()

,Name,Link,Image,Rating,Reviews,Price
0,Tapo c200 360-degree smart wi-fi pan and tilt ...,https://www.amazon.eg/-/en/Tapo-360-degree-sma...,https://m.media-amazon.com/images/I/41a5w8F1xG...,4.6,"150,327 ratings",888
1,"Liphisy 64” Tripod for Cell Phone & Camera, Ph...",https://www.amazon.eg/-/en/Liphisy-Tripod-Adju...,https://m.media-amazon.com/images/I/61G1zIdlFw...,4.0,"6,945 ratings",475
2,Kingston Canvas Select Plus microSD Memory Car...,https://www.amazon.eg/-/en/Kingston-microSD-12...,https://m.media-amazon.com/images/I/71913emXQO...,4.4,"2,465 ratings","1,179"
3,EZVIZ CS-C6N 1080p HD Indoor Wi-Fi Pan/Tilt Se...,https://www.amazon.eg/-/en/EZVIZ-CS-C6N-Indoor...,https://m.media-amazon.com/images/I/51Uyc1UK5u...,4.3,"7,501 ratings",899
4,Tapo TP-Link 2K Pan Tilt Security Camera for B...,https://www.amazon.eg/-/en/Tapo-TP-Link-Securi...,https://m.media-amazon.com/images/I/61diChCuq+...,4.5,"10,958 ratings",949


In [7]:
df_Cameras.to_csv("amazon_eg_cameras.csv", index=False, encoding="utf-8-sig")

In [8]:
df = pd.read_csv("amazon_eg_cameras.csv", encoding="utf-8-sig")

In [9]:
df

,Name,Link,Image,Rating,Reviews,Price
0,Tapo c200 360-degree smart wi-fi pan and tilt ...,https://www.amazon.eg/-/en/Tapo-360-degree-sma...,https://m.media-amazon.com/images/I/41a5w8F1xG...,4.6,"150,327 ratings",888
1,"Liphisy 64” Tripod for Cell Phone & Camera, Ph...",https://www.amazon.eg/-/en/Liphisy-Tripod-Adju...,https://m.media-amazon.com/images/I/61G1zIdlFw...,4.0,"6,945 ratings",475
2,Kingston Canvas Select Plus microSD Memory Car...,https://www.amazon.eg/-/en/Kingston-microSD-12...,https://m.media-amazon.com/images/I/71913emXQO...,4.4,"2,465 ratings","1,179"
3,EZVIZ CS-C6N 1080p HD Indoor Wi-Fi Pan/Tilt Se...,https://www.amazon.eg/-/en/EZVIZ-CS-C6N-Indoor...,https://m.media-amazon.com/images/I/51Uyc1UK5u...,4.3,"7,501 ratings",899
4,Tapo TP-Link 2K Pan Tilt Security Camera for B...,https://www.amazon.eg/-/en/Tapo-TP-Link-Securi...,https://m.media-amazon.com/images/I/61diChCuq+...,4.5,"10,958 ratings",949
...,...,...,...,...,...,...
4542,Godox Litemons LP400R 36W RGB LED Panel Video ...,https://www.amazon.eg/-/en/Litemons-LP400R-Bar...,https://m.media-amazon.com/images/I/71HqKPOXAs...,4.8,15 ratings,"4,999"
4543,Jmary KP-2234 Professional Camera Tripod - Bla...,https://www.amazon.eg/-/en/Jmary-KP-2234-Profe...,https://m.media-amazon.com/images/I/61ZcDO9v5V...,Not Available,Not Available,"1,455"
4544,Elfcam® - 100m/328.08ft 8K HDMI 2.1 Cable Fibe...,https://www.amazon.eg/-/en/Elfcam%C2%AE-328-08...,https://m.media-amazon.com/images/I/51WG8Oz9+7...,4.4,47 ratings,"9,000"
4545,eufy Security eufyCam S3 Pro Add-on Solar Secu...,https://www.amazon.eg/-/en/eufy-Security-Resol...,https://m.media-amazon.com/images/I/510iHtzrNt...,3.9,30 ratings,"14,349"


In [1]:
import pandas as pd
df = pd.read_csv("amazon_eg_cameras.csv", encoding="utf-8-sig")
df.to_parquet("amazon_eg_cameras.parquet", index=False)